# DA6401 Report - Section 2.3 Optimizer Showdown

This notebook runs the controlled optimizer comparison for:
- SGD
- Momentum
- NAG
- RMSProp

All runs use identical architecture/hyperparameters except optimizer.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import wandb

import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

print("Project root:", ROOT)


In [ ]:
from train import train_and_evaluate

RUN_EXPERIMENT = False  # Set to True when you want to launch W&B runs.

PROJECT = "da6401_assignment_1"
ENTITY = None
RUN_GROUP = "report_v1_2_3_optimizer_showdown"

common = dict(
    dataset="mnist",
    epochs=10,
    batch_size=64,
    loss="cross_entropy",
    learning_rate=0.001,
    weight_decay=0.0,
    num_layers=3,
    hidden_size=[128, 128, 128],
    activation="relu",
    weight_init="xavier",
    wandb_project=PROJECT,
    wandb_entity=ENTITY,
    wandb_mode="online",
    seed=42,
)

optimizers = ["sgd", "momentum", "nag", "rmsprop"]
out_dir = ROOT / "src" / "tmp"
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
results = []

if RUN_EXPERIMENT:
    for opt in optimizers:
        args = SimpleNamespace(
            optimizer=opt,
            model_path=str(out_dir / f"model_2_3_{opt}.npy"),
            config_path=str(out_dir / f"config_2_3_{opt}.json"),
            **common,
        )

        run = wandb.init(
            project=PROJECT,
            entity=ENTITY,
            name=f"report_2_3_optimizer_{opt}",
            group=RUN_GROUP,
            tags=["report_v1", "report_section_2.3", "optimizer_showdown", "mnist", opt],
            config=vars(args),
        )

        result = train_and_evaluate(args, wandb_run_override=run)
        history = result["history"]

        train_losses = [float(ep["train"]["loss"]) for ep in history]
        val_losses = [float(ep["val"]["loss"]) for ep in history]

        rec = {
            "optimizer": opt,
            "run_id": run.id,
            "run_name": run.name,
            "train_loss_epoch_1_to_5": train_losses[:5],
            "val_loss_epoch_1_to_5": val_losses[:5],
            "train_loss_epoch_5": train_losses[4],
            "val_loss_epoch_5": val_losses[4],
            "train_drop_1_to_5": train_losses[0] - train_losses[4],
            "val_drop_1_to_5": val_losses[0] - val_losses[4],
            "final_val_f1": float(result["final_metrics"]["val"]["f1"]),
            "final_test_f1": float(result["final_metrics"]["test"]["f1"]),
        }
        results.append(rec)

        run.summary["analysis/train_loss_epoch_5"] = rec["train_loss_epoch_5"]
        run.summary["analysis/val_loss_epoch_5"] = rec["val_loss_epoch_5"]
        run.summary["analysis/train_drop_1_to_5"] = rec["train_drop_1_to_5"]
        run.summary["analysis/val_drop_1_to_5"] = rec["val_drop_1_to_5"]
        run.finish()

    payload = {
        "setup": {
            "dataset": "mnist",
            "epochs": 10,
            "batch_size": 64,
            "loss": "cross_entropy",
            "learning_rate": 0.001,
            "weight_decay": 0.0,
            "architecture": "3 hidden layers, 128 neurons each, ReLU",
            "weight_init": "xavier",
            "seed": 42,
        },
        "runs": results,
        "fastest_first5_by_train_loss": min(results, key=lambda x: x["train_loss_epoch_5"]),
        "fastest_first5_by_val_loss": min(results, key=lambda x: x["val_loss_epoch_5"]),
    }

    with open(out_dir / "report_2_3_optimizer_showdown.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print("Completed 2.3 runs")
else:
    print("RUN_EXPERIMENT is False. No runs were launched.")


In [ ]:
summary_path = out_dir / "report_2_3_optimizer_showdown.json"
if summary_path.exists():
    data = json.loads(summary_path.read_text())
    print("Fastest by epoch-5 train loss:", data["fastest_first5_by_train_loss"]["optimizer"])
    print("Fastest by epoch-5 val loss:", data["fastest_first5_by_val_loss"]["optimizer"])
    for r in data["runs"]:
        print(r["optimizer"], "train_e5", round(r["train_loss_epoch_5"], 6), "val_e5", round(r["val_loss_epoch_5"], 6), "run", r["run_id"])
else:
    print("No summary file yet. Run the experiment cell first.")
